> **Actualizaci?n del 01-Oct-2026:** los bloques siguientes documentan la versi?n anterior a la refactorizaci?n. La implementaci?n vigente utiliza servicios por m?dulo, httpx/asyncio y vistas reutilizables. Consulte los comentarios/docstrings en `frontend/src/services/`, `frontend/src/app/async_runner.py` y `frontend/src/ui/views/`, y la secci?n 4.1 de `Plan_Proyecto.ipynb`. El generador anterior no est? presente en el workspace actual; no se regeneran estas instant?neas autom?ticamente.


# Documentación detallada del front-end Fixify

Fecha de revisión: **1 de octubre de 2026**. Documentación del código existente, previa a implementar el plan. La interfaz es una aplicación **de escritorio Python/Tkinter/ttkbootstrap**, no una aplicación web ni Flutter.

## Cómo usar este notebook

Lee las explicaciones y las referencias de código en orden. El código de la aplicación aparece en bloques Markdown y **no se ejecuta** al usar Run All. Las únicas celdas ejecutables inspeccionan archivos locales; no abren ventanas, no hacen HTTP y no acceden a Supabase. Las salidas se dejan vacías para evitar incluir datos privados. Cada archivo incluye huella SHA-256 para identificar la versión documentada.

## Arquitectura y recorrido

```text
app/main.py → VentanaLogin → APIService → FastAPI → PostgreSQL en Supabase
                    ↓ login exitoso
              SistemaGestion → dashboard / clientes / equipos / usuarios
```

Tkinter aporta el bucle de eventos y ttkbootstrap los controles/tema. Requests transporta JSON y formularios a la API. El front-end no usa credenciales de BD ni Supabase Auth. `models/database.py` es un prototipo sin conexión con el login activo.

## Contrato consumido

| Método | Ruta | Envío / resultado | Permiso backend actual |
|---|---|---|---|
| POST | /auth/login | Formulario username/password → access_token, token_type, role, message | Público; cuenta activa |
| GET / POST | /usuarios/ | Lista de usuarios / JSON username,password,role | admin |
| GET / POST | /clientes/ | Lista de clientes / JSON nombre,telefono,correo,direccion | admin, recepcion |
| GET | /equipos/ | UI espera lista con id,tipo,marca,modelo,numero_serie,problema_reportado,client_id | **Ruta ausente** |
| POST | /equipos/ | JSON de equipo → mensaje,folio,detalles | admin, recepcion |

Datos opcionales vacíos se convierten a null. El token viaja solo en cabecera Authorization. No registrar contraseña, JWT ni cabeceras en las evidencias.

## Requisitos y pendientes observados

R01=RF-01: login existe; hay que validar ejecución real, rechazo de credenciales y cierre que limpie token. R02=RF-02: hay menús condicionados y autorización servidor, pero faltan acciones completas de HU-02 y coherencia de navegación por rol. R03=RF-03: formulario integrado con API; persistencia real pendiente de prueba. R04=RF-04: formulario y asociación existen, pero GET ausente y FK incompatible impiden dar por cerrado el recorrido.

Las llamadas HTTP bloquean UI durante la espera; dashboard puede solicitar tres llamadas de cinco segundos consecutivas y ocultar errores como ceros. Proponer trabajo en segundo plano con actualización de widgets en el hilo de Tkinter y estado de carga. Mantener separado el rol de dominio y la etiqueta visible.

## Ejecución de desarrollo

Desde la raíz: `python frontend/src/app/main.py`, con backend disponible en la URL del cliente. Dependencias identificadas en imports: requests, ttkbootstrap y tkinter (incluido según instalación Python). No existe manifiesto de dependencias específico de frontend en el repositorio revisado; capturar versiones probadas antes de fijarlas. No se instalaron dependencias ni se abrió la GUI para esta documentación.


## Inventario detallado por archivo

Las líneas corresponden al archivo original. Los bloques siguientes son referencias de lectura; las explicaciones describen el comportamiento actual y sus límites.

### `frontend/src/app/__init__.py`

Marcador de paquete Python. Archivo vacío en la revisión: no inicializa servicios ni ejecuta lógica propia.

SHA-256 del original: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.

No define clases ni funciones.


### `frontend/src/app/main.py`

Punto de entrada de escritorio. Calcula `RUTA_SRC` a partir de `__file__` y la inserta en `sys.path` para resolver los imports `ui` y `services`. Solo al ejecutar el archivo como programa crea `VentanaLogin` y entra en `mainloop()`. Tkinter procesa los eventos en ese bucle; los callbacks HTTP actuales se ejecutan en el mismo hilo.

SHA-256 del original: `8a9d20236e99dfbb681f046891ff29a87c6c7dfe94e0d1b68f8b165afcff7ebc`.

**Dependencias importadas**

```python
import sys
import os
from ui.login import VentanaLogin
```

No define clases ni funciones.


**Código de referencia (no ejecutable)**

```python
# app/main.py

import sys
import os

# Obtener la ruta absoluta de la carpeta 'src' y ponerla al inicio de la búsqueda
RUTA_SRC = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
if RUTA_SRC not in sys.path:
    sys.path.insert(0, RUTA_SRC)

from ui.login import VentanaLogin

if __name__ == "__main__":
    app = VentanaLogin()
    app.mainloop()
```

### `frontend/src/models/__init__.py`

Marcador de paquete Python. Archivo vacío en la revisión: no inicializa servicios ni ejecuta lógica propia.

SHA-256 del original: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.

No define clases ni funciones.


### `frontend/src/models/database.py`

Prototipo de usuarios en memoria, con nombre, contraseña, rol y estado. No es Supabase ni el modelo activo de autenticación: la ventana de login usa `api_client`. Los valores de contraseñas de demostración se omiten en esta documentación. Mantener este archivo puede confundir al equipo sobre la fuente de datos; revisar su retirada después de confirmar que no tiene consumidores externos.

SHA-256 del original: `9eb4cd724e929b30519c145a718ca9dfaa0d828a11efd2819d42bf8ad52b8b56`.

No define clases ni funciones.


**Código de referencia (no ejecutable)**

```python
# models/database.py

# Base de datos simulada en memoria
USUARIOS_BD = {
    "admin": {"nombre": "Administrador Principal", "password": "<omitida>", "rol": "Administrador", "estado": "Activo"},
    "tecnico": {"nombre": "Técnico Soporte", "password": "<omitida>", "rol": "Técnico", "estado": "Activo"},
    "recepcion": {"nombre": "Recepción Turno 1", "password": "<omitida>", "rol": "Recepción", "estado": "Activo"}
}
```

### `frontend/src/services/api_service.py`

`APIService` concentra las solicitudes HTTP y existe una instancia global `api_client`. Su URL predeterminada es `http://127.0.0.1:8000`; no se lee de un archivo de configuración. Guarda el JWT en memoria con `set_token` y construye `Authorization: Bearer …` mediante `_headers`. El login envía un formulario (`data`), mientras las altas envían JSON. Cada operación devuelve `(éxito, datos_o_error)`, no una excepción de negocio a la interfaz. `_parse_error` convierte `detail` de FastAPI en texto y une las listas de errores Pydantic; si la respuesta no es JSON utiliza el texto de respuesta. Todas las llamadas tienen `timeout=5` y capturan excepciones de forma amplia.

`login` guarda el token al recibir 200; `obtener_usuarios`, `obtener_clientes` y `obtener_equipos` aceptan 200; las tres altas aceptan 200 o 201. `crear_cliente` transforma correo/dirección vacíos en `None`. `crear_equipo` convierte `client_id` a entero antes del bloque `try`, por lo que una conversión inválida escapa al manejo de errores. No existen métodos para activar/desactivar usuarios, cambiar roles, buscar clientes ni limpiar la sesión. La llamada GET de equipos no tiene contraparte en el backend actual.

SHA-256 del original: `f171f6914a441e8002d2cf28e77f779edcdc2a1aaa531670fa7a615d878ba699`.

**Dependencias importadas**

```python
import requests
```

**Clases y funciones, incluidas funciones anidadas**

| Símbolo | Líneas | Firma | Decoradores |
|---|---|---|---|
| `APIService` | 3–116 | `class APIService()` | `—` |
| `APIService.__init__` | 4–6 | `__init__(self, base_url='http://127.0.0.1:8000')` | `—` |
| `APIService.set_token` | 8–9 | `set_token(self, token)` | `—` |
| `APIService._headers` | 11–15 | `_headers(self)` | `—` |
| `APIService._parse_error` | 17–28 | `_parse_error(self, res)` | `—` |
| `APIService.login` | 30–41 | `login(self, username, password)` | `—` |
| `APIService.obtener_usuarios` | 43–51 | `obtener_usuarios(self)` | `—` |
| `APIService.crear_usuario` | 53–62 | `crear_usuario(self, username, password, role)` | `—` |
| `APIService.obtener_clientes` | 64–72 | `obtener_clientes(self)` | `—` |
| `APIService.crear_cliente` | 74–88 | `crear_cliente(self, nombre, telefono, correo=None, direccion=None)` | `—` |
| `APIService.obtener_equipos` | 90–98 | `obtener_equipos(self)` | `—` |
| `APIService.crear_equipo` | 100–116 | `crear_equipo(self, tipo, marca, modelo, problema_reportado, client_id, numero_serie=None)` | `—` |


**Código de referencia (no ejecutable)**

```python
import requests

class APIService:
    def __init__(self, base_url="http://127.0.0.1:8000"):
        self.base_url = base_url
        self.token = None

    def set_token(self, token):
        self.token = token

    def _headers(self):
        headers = {}
        if self.token:
            headers["Authorization"] = f"Bearer {self.token}"
        return headers

    def _parse_error(self, res):
        """Formatea adecuadamente respuestas de error de FastAPI (strings o listas Pydantic)."""
        try:
            data = res.json()
            detail = data.get("detail", res.text)
            if isinstance(detail, list):
                # Errores de validación Pydantic
                msgs = [f"• {err.get('msg', 'Error en el campo ' + str(err.get('loc')))}" for err in detail]
                return "\n".join(msgs)
            return str(detail)
        except Exception:
            return res.text or "Error desconocido en el servidor"

    def login(self, username, password):
        url = f"{self.base_url}/auth/login"
        payload = {"username": username, "password": password}
        try:
            res = requests.post(url, data=payload, timeout=5)
            if res.status_code == 200:
                data = res.json()
                self.token = data.get("access_token")
                return True, data
            return False, self._parse_error(res)
        except Exception as e:
            return False, f"No se pudo conectar con el servidor: {str(e)}"

    def obtener_usuarios(self):
        url = f"{self.base_url}/usuarios/"
        try:
            res = requests.get(url, headers=self._headers(), timeout=5)
            if res.status_code == 200:
                return True, res.json()
            return False, self._parse_error(res)
        except Exception as e:
            return False, str(e)

    def crear_usuario(self, username, password, role):
        url = f"{self.base_url}/usuarios/"
        payload = {"username": username, "password": password, "role": role}
        try:
            res = requests.post(url, json=payload, headers=self._headers(), timeout=5)
            if res.status_code in (200, 201):
                return True, res.json()
            return False, self._parse_error(res)
        except Exception as e:
            return False, str(e)

    def obtener_clientes(self):
        url = f"{self.base_url}/clientes/"
        try:
            res = requests.get(url, headers=self._headers(), timeout=5)
            if res.status_code == 200:
                return True, res.json()
            return False, self._parse_error(res)
        except Exception as e:
            return False, str(e)

    def crear_cliente(self, nombre, telefono, correo=None, direccion=None):
        url = f"{self.base_url}/clientes/"
        payload = {
            "nombre": nombre,
            "telefono": telefono,
            "correo": correo if correo else None,
            "direccion": direccion if direccion else None
        }
        try:
            res = requests.post(url, json=payload, headers=self._headers(), timeout=5)
            if res.status_code in (200, 201):
                return True, res.json()
            return False, self._parse_error(res)
        except Exception as e:
            return False, str(e)

    def obtener_equipos(self):
        url = f"{self.base_url}/equipos/"
        try:
            res = requests.get(url, headers=self._headers(), timeout=5)
            if res.status_code == 200:
                return True, res.json()
            return False, self._parse_error(res)
        except Exception as e:
            return False, str(e)

    def crear_equipo(self, tipo, marca, modelo, problema_reportado, client_id, numero_serie=None):
        url = f"{self.base_url}/equipos/"
        payload = {
            "tipo": tipo,
            "marca": marca,
            "modelo": modelo,
            "numero_serie": numero_serie if numero_serie else None,
            "problema_reportado": problema_reportado,
            "client_id": int(client_id)
        }
        try:
            res = requests.post(url, json=payload, headers=self._headers(), timeout=5)
            if res.status_code in (200, 201):
                return True, res.json()
            return False, self._parse_error(res)
        except Exception as e:
            return False, str(e)

api_client = APIService()
```

### `frontend/src/ui/__init__.py`

Marcador de paquete Python. Archivo vacío en la revisión: no inicializa servicios ni ejecuta lógica propia.

SHA-256 del original: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.

No define clases ni funciones.


### `frontend/src/ui/login.py`

`VentanaLogin` hereda de `ttk.Window`, utiliza el tema `litera` y presenta usuario y contraseña con placeholders. `crear_widgets` registra callbacks de foco, login y recuperación. Los cuatro helpers de placeholder borran/restauran texto y alternan el enmascarado de contraseña.

`verificar_login` obtiene el usuario con `strip().lower()` y la contraseña con `strip()`, llama a la API y, si tiene éxito, traduce roles de dominio (`admin`, `tecnico`, `recepcion`) a etiquetas visibles. Oculta la ventana con `withdraw`, limpia campos e importa `SistemaGestion` dentro del callback. El `master` conserva la ventana de login para volver a ella. Si falla muestra `messagebox`. Hay un defecto: después de convertir a minúsculas, compara el placeholder de usuario con `USUARIO` en mayúsculas; además no conviene modificar la contraseña con `strip`. `mostrar_recuperacion` únicamente informa que se contacte al administrador, sin endpoint ni envío de recuperación.

SHA-256 del original: `53c461d19d12a607490bea52a9e69f322b21e1d218fd213219045b838178f6a6`.

**Dependencias importadas**

```python
import ttkbootstrap as ttk
from ttkbootstrap.constants import *
from tkinter import messagebox
from services.api_service import api_client
from ui.sistema_gestion import SistemaGestion
```

**Clases y funciones, incluidas funciones anidadas**

| Símbolo | Líneas | Firma | Decoradores |
|---|---|---|---|
| `VentanaLogin` | 6–126 | `class VentanaLogin(ttk.Window)` | `—` |
| `VentanaLogin.__init__` | 7–17 | `__init__(self)` | `—` |
| `VentanaLogin.crear_widgets` | 19–51 | `crear_widgets(self)` | `—` |
| `VentanaLogin.quitar_placeholder` | 53–55 | `quitar_placeholder(self, entry, texto)` | `—` |
| `VentanaLogin.poner_placeholder` | 57–59 | `poner_placeholder(self, entry, texto)` | `—` |
| `VentanaLogin.activar_password_mode` | 61–64 | `activar_password_mode(self, entry, texto)` | `—` |
| `VentanaLogin.poner_placeholder_pass` | 66–69 | `poner_placeholder_pass(self, entry, texto)` | `—` |
| `VentanaLogin.verificar_login` | 71–103 | `verificar_login(self)` | `—` |
| `VentanaLogin.mostrar_recuperacion` | 105–126 | `mostrar_recuperacion(self)` | `—` |


**Código de referencia (no ejecutable)**

```python
import ttkbootstrap as ttk
from ttkbootstrap.constants import *
from tkinter import messagebox
from services.api_service import api_client

class VentanaLogin(ttk.Window):
    def __init__(self):
        super().__init__(
            title="Iniciar Sesión - Sistema de Gestión",
            themename="litera",
            size=(450, 460),
            minsize=(400, 450)
        )
        self.resizable(False, False)
        self.place_window_center()

        self.crear_widgets()

    def crear_widgets(self):
        main_frame = ttk.Frame(self, padding=35)
        main_frame.pack(fill=BOTH, expand=YES)

        ttk.Label(main_frame, text="SISTEMA DE GESTIÓN", font=("Arial", 16, "bold")).pack(pady=(0, 5))
        ttk.Label(main_frame, text="DE EQUIPOS TECNOLÓGICOS", font=("Arial", 10)).pack(pady=(0, 25))
        
        self.entry_usuario = ttk.Entry(main_frame, width=32)
        self.entry_usuario.insert(0, "USUARIO")
        self.entry_usuario.pack(pady=10, ipady=4)
        self.entry_usuario.bind("<FocusIn>", lambda e: self.quitar_placeholder(self.entry_usuario, "USUARIO"))
        self.entry_usuario.bind("<FocusOut>", lambda e: self.poner_placeholder(self.entry_usuario, "USUARIO"))

        self.entry_password = ttk.Entry(main_frame, width=32)
        self.entry_password.insert(0, "CONTRASEÑA")
        self.entry_password.pack(pady=10, ipady=4)
        self.entry_password.bind("<FocusIn>", lambda e: self.activar_password_mode(self.entry_password, "CONTRASEÑA"))
        self.entry_password.bind("<FocusOut>", lambda e: self.poner_placeholder_pass(self.entry_password, "CONTRASEÑA"))

        ttk.Button(
            main_frame, 
            text="Iniciar Sesión", 
            bootstyle="secondary", 
            command=self.verificar_login, 
            width=28
        ).pack(pady=(20, 10), ipady=4)

        ttk.Button(
            main_frame, 
            text="¿Olvidaste tu contraseña?", 
            bootstyle="link", 
            command=self.mostrar_recuperacion
        ).pack(pady=(5, 0))

    def quitar_placeholder(self, entry, texto):
        if entry.get() == texto:
            entry.delete(0, END)

    def poner_placeholder(self, entry, texto):
        if not entry.get():
            entry.insert(0, texto)

    def activar_password_mode(self, entry, texto):
        if entry.get() == texto:
            entry.delete(0, END)
            entry.config(show="*")

    def poner_placeholder_pass(self, entry, texto):
        if not entry.get():
            entry.config(show="")
            entry.insert(0, texto)

    def verificar_login(self):
        usuario = self.entry_usuario.get().strip().lower()
        password = self.entry_password.get().strip()

        if not usuario or usuario == "USUARIO" or not password or password == "CONTRASEÑA":
            messagebox.showerror("Error de acceso", "Por favor, complete los campos de usuario y contraseña.")
            return

        exito, data = api_client.login(usuario, password)

        if exito:
            token = data.get("access_token")
            rol_backend = data.get("role", "tecnico")
            
            mapa_roles = {
                "admin": "Administrador",
                "tecnico": "Técnico",
                "recepcion": "Recepción"
            }
            rol_display = mapa_roles.get(rol_backend, rol_backend.capitalize())

            self.withdraw()
            
            self.entry_usuario.delete(0, 'end')
            self.poner_placeholder(self.entry_usuario, "USUARIO")
            self.entry_password.delete(0, 'end')
            self.poner_placeholder_pass(self.entry_password, "CONTRASEÑA")
            self.focus_set()
            
            from ui.sistema_gestion import SistemaGestion
            SistemaGestion(master=self, usuario_actual=usuario, rol_actual=rol_display, token=token)
        else:
            messagebox.showerror("Acceso denegado", str(data))

    def mostrar_recuperacion(self):
        ventana = ttk.Toplevel(self)
        ventana.title("Recuperar Cuenta")
        ventana.geometry("480x280")
        ventana.resizable(False, False)
        ventana.place_window_center()

        ttk.Label(ventana, text="Recuperación de Cuenta", font=("Arial", 16, "bold")).pack(pady=(20, 10))

        mensaje = (
            "Si no recuerdas tu contraseña o necesitas acceso al sistema, "
            "contacta al administrador de la plataforma para restablecer tu cuenta."
        )
        ttk.Label(ventana, text=mensaje, wraplength=400, justify=CENTER).pack(pady=20)

        ttk.Button(
            ventana, 
            text="Entendido", 
            bootstyle="secondary", 
            command=ventana.destroy, 
            width=20
        ).pack(pady=10)
```

### `frontend/src/ui/sistema_gestion.py`

`SistemaGestion` es una ventana `Toplevel` que conserva usuario, etiqueta de rol y token; vuelve a establecer el token global y construye sidebar y área de contenido. `crear_layout` muestra Dashboard, Equipos y Clientes a todos los roles, y Configuración solo al administrador. Las altas de cliente y equipo se muestran para administración/recepción. Esto es presentación: la API debe autorizar cada petición.

`cerrar_aplicacion` destruye la ventana y su master; `cerrar_sesion` pide confirmación, destruye solo la ventana de gestión y vuelve a mostrar login, pero no borra el token global. `limpiar_contenido` destruye los widgets antes de cambiar de vista y `crear_encabezado` agrega su título.

`mostrar_dashboard` realiza tres GET consecutivos y calcula conteos con `len`. Los fallos se convierten en cero, por lo que cero no siempre significa ausencia de registros. Muestra los últimos ocho elementos de la lista de equipos en orden inverso, sin ordenación temporal garantizada por API. `crear_tarjeta` compone las tarjetas. El GET de usuarios exige admin, pero también se pide para recepción/técnico. El GET de equipos no está implementado.

`mostrar_equipos`, `mostrar_clientes` y `mostrar_configuracion` crean tablas Treeview; `cargar_equipos`, `cargar_clientes` y `cargar_usuarios` vacían y rellenan sus filas o muestran error. Usuarios transforma `is_active` a estado visible. Los folios de equipos son `EQ-{id:03d}` calculados en UI; no representan órdenes de servicio.

`modal_nuevo_cliente` captura nombre/teléfono obligatorios y correo/dirección opcionales. Su `guardar` aplica `strip`, valida presencia, envía datos y recarga la tabla. `modal_nuevo_equipo` primero consulta clientes: sin clientes disponibles no permite registrar. Construye un mapa etiqueta→ID y un selector readonly, captura tipo, marca, modelo, serie opcional y problema; su `guardar` valida presencia, solicita alta, muestra folio y vuelve a la vista de equipos. Esa recarga fallará mientras falte GET. `modal_nuevo_usuario` presenta roles con etiquetas, su `guardar` los convierte a valores de dominio y crea la cuenta. Las longitudes/formato se validan principalmente en backend. Los tres `guardar` son funciones anidadas que capturan los controles de cada modal.

Las solicitudes síncronas bloquean el hilo de interfaz; no hay indicador de carga, paginación, búsqueda integrada ni controles de activación/desactivación/cambio de rol. La visibilidad de menús y los permisos reales están desalineados para técnicos.

SHA-256 del original: `23978a4504369ad14bb88911049695e989bcb12acae3e4996d68de70ef37e7a2`.

**Dependencias importadas**

```python
import ttkbootstrap as ttk
from ttkbootstrap.constants import *
from tkinter import messagebox
from services.api_service import api_client
```

**Clases y funciones, incluidas funciones anidadas**

| Símbolo | Líneas | Firma | Decoradores |
|---|---|---|---|
| `SistemaGestion` | 6–400 | `class SistemaGestion(ttk.Toplevel)` | `—` |
| `SistemaGestion.__init__` | 7–25 | `__init__(self, master=None, usuario_actual='Invitado', rol_actual='invitado', token=None)` | `—` |
| `SistemaGestion.cerrar_aplicacion` | 27–30 | `cerrar_aplicacion(self)` | `—` |
| `SistemaGestion.cerrar_sesion` | 32–36 | `cerrar_sesion(self)` | `—` |
| `SistemaGestion.crear_layout` | 38–72 | `crear_layout(self)` | `—` |
| `SistemaGestion.limpiar_contenido` | 74–76 | `limpiar_contenido(self)` | `—` |
| `SistemaGestion.crear_encabezado` | 78–80 | `crear_encabezado(self, titulo)` | `—` |
| `SistemaGestion.mostrar_dashboard` | 83–125 | `mostrar_dashboard(self)` | `—` |
| `SistemaGestion.crear_tarjeta` | 127–133 | `crear_tarjeta(self, parent, titulo, valor, estilo, col)` | `—` |
| `SistemaGestion.mostrar_equipos` | 136–153 | `mostrar_equipos(self)` | `—` |
| `SistemaGestion.cargar_equipos` | 155–173 | `cargar_equipos(self)` | `—` |
| `SistemaGestion.mostrar_clientes` | 176–193 | `mostrar_clientes(self)` | `—` |
| `SistemaGestion.cargar_clientes` | 195–210 | `cargar_clientes(self)` | `—` |
| `SistemaGestion.mostrar_configuracion` | 213–229 | `mostrar_configuracion(self)` | `—` |
| `SistemaGestion.cargar_usuarios` | 231–243 | `cargar_usuarios(self)` | `—` |
| `SistemaGestion.modal_nuevo_cliente` | 246–288 | `modal_nuevo_cliente(self)` | `—` |
| `SistemaGestion.modal_nuevo_cliente.guardar` | 270–286 | `guardar()` | `—` |
| `SistemaGestion.modal_nuevo_equipo` | 290–353 | `modal_nuevo_equipo(self)` | `—` |
| `SistemaGestion.modal_nuevo_equipo.guardar` | 331–351 | `guardar()` | `—` |
| `SistemaGestion.modal_nuevo_usuario` | 355–400 | `modal_nuevo_usuario(self)` | `—` |
| `SistemaGestion.modal_nuevo_usuario.guardar` | 376–398 | `guardar()` | `—` |


**Código de referencia (no ejecutable)**

```python
import ttkbootstrap as ttk
from ttkbootstrap.constants import *
from tkinter import messagebox
from services.api_service import api_client

class SistemaGestion(ttk.Toplevel):
    def __init__(self, master=None, usuario_actual="Invitado", rol_actual="invitado", token=None):
        super().__init__(master)
        self.master = master
        self.usuario_actual = usuario_actual
        self.rol_actual = rol_actual
        self.token = token
        
        if self.token:
            api_client.set_token(self.token)

        self.title("Sistema de Gestión de Equipos Tecnológicos - Fixify")
        self.geometry("1100x700")
        self.minsize(900, 600)
        
        self.protocol("WM_DELETE_WINDOW", self.cerrar_aplicacion)
        self.place_window_center()

        self.crear_layout()
        self.mostrar_dashboard()

    def cerrar_aplicacion(self):
        self.destroy()
        if self.master:
            self.master.destroy()

    def cerrar_sesion(self):
        if messagebox.askyesno("Cerrar Sesión", "¿Está seguro de que desea salir del sistema?"):
            self.destroy()
            if self.master:
                self.master.deiconify()

    def crear_layout(self):
        self.grid_rowconfigure(0, weight=1)
        self.grid_columnconfigure(1, weight=1)

        self.sidebar = ttk.Frame(self, bootstyle="secondary", padding=15)
        self.sidebar.grid(row=0, column=0, sticky="nsew")

        lbl_titulo = ttk.Label(self.sidebar, text="FIXIFY", font=("Arial", 18, "bold"), bootstyle="inverse-secondary")
        lbl_titulo.pack(pady=(10, 20), anchor="w")

        info_user = f"Usuario: {self.usuario_actual.upper()}\nRol: {self.rol_actual}"
        lbl_user = ttk.Label(self.sidebar, text=info_user, font=("Arial", 9), bootstyle="inverse-secondary")
        lbl_user.pack(pady=(0, 25), anchor="w")

        btn_dash = ttk.Button(self.sidebar, text="📊  Dashboard", bootstyle="secondary", command=self.mostrar_dashboard, width=18)
        btn_dash.pack(pady=5, fill=X)

        btn_equipos = ttk.Button(self.sidebar, text="💻  Equipos", bootstyle="secondary", command=self.mostrar_equipos, width=18)
        btn_equipos.pack(pady=5, fill=X)

        btn_clientes = ttk.Button(self.sidebar, text="👥  Clientes", bootstyle="secondary", command=self.mostrar_clientes, width=18)
        btn_clientes.pack(pady=5, fill=X)

        if self.rol_actual.lower() in ["administrador", "admin"]:
            btn_config = ttk.Button(self.sidebar, text="⚙️  Configuración", bootstyle="secondary", command=self.mostrar_configuracion, width=18)
            btn_config.pack(pady=5, fill=X)

        frame_bottom = ttk.Frame(self.sidebar, bootstyle="secondary")
        frame_bottom.pack(side=BOTTOM, fill=X, pady=10)

        btn_logout = ttk.Button(frame_bottom, text="🚪  Cerrar Sesión", bootstyle="danger", command=self.cerrar_sesion, width=18)
        btn_logout.pack(fill=X)

        self.content_area = ttk.Frame(self, padding=25)
        self.content_area.grid(row=0, column=1, sticky="nsew")

    def limpiar_contenido(self):
        for widget in self.content_area.winfo_children():
            widget.destroy()

    def crear_encabezado(self, titulo):
        lbl = ttk.Label(self.content_area, text=titulo, font=("Arial", 18, "bold"))
        lbl.pack(anchor="w", pady=(0, 20))

    # --- DASHBOARD DINÁMICO ---
    def mostrar_dashboard(self):
        self.limpiar_contenido()
        self.crear_encabezado("Dashboard General")

        # Consultamos datos reales a la base de datos
        exito_eq, equipos = api_client.obtener_equipos()
        exito_cli, clientes = api_client.obtener_clientes()
        exito_usr, usuarios = api_client.obtener_usuarios()

        total_equipos = len(equipos) if (exito_eq and isinstance(equipos, list)) else 0
        total_clientes = len(clientes) if (exito_cli and isinstance(clientes, list)) else 0
        total_usuarios = len(usuarios) if (exito_usr and isinstance(usuarios, list)) else 0

        frame_cards = ttk.Frame(self.content_area)
        frame_cards.pack(fill=X, pady=(0, 20))

        # Tarjetas dinámicas que incrementan en tiempo real
        self.crear_tarjeta(frame_cards, "Total Equipos", str(total_equipos), "info", 0)
        self.crear_tarjeta(frame_cards, "Clientes Registrados", str(total_clientes), "success", 1)
        self.crear_tarjeta(frame_cards, "Usuarios / Personal", str(total_usuarios), "primary", 2)

        lbl_recent = ttk.Label(self.content_area, text="Últimos Equipos Registrados", font=("Arial", 12, "bold"))
        lbl_recent.pack(anchor="w", pady=(10, 10))

        cols = ("Folio", "Tipo", "Marca", "Modelo", "Problema", "ID Cliente")
        tabla = ttk.Treeview(self.content_area, columns=cols, show="headings", height=8)
        for col in cols:
            tabla.heading(col, text=col)
            tabla.column(col, anchor=CENTER, width=130)
        tabla.pack(fill=BOTH, expand=YES)

        # Cargar actividades recientes
        if exito_eq and isinstance(equipos, list):
            for eq in reversed(equipos[-8:]):
                folio = f"EQ-{eq.get('id', 0):03d}"
                tabla.insert("", END, values=(
                    folio,
                    eq.get("tipo", ""),
                    eq.get("marca", ""),
                    eq.get("modelo", ""),
                    eq.get("problema_reportado", ""),
                    eq.get("client_id", "")
                ))

    def crear_tarjeta(self, parent, titulo, valor, estilo, col):
        card = ttk.Labelframe(parent, text=titulo, bootstyle=estilo, padding=15)
        card.grid(row=0, column=col, padx=10, pady=5, sticky="nsew")
        parent.grid_columnconfigure(col, weight=1)

        lbl_val = ttk.Label(card, text=valor, font=("Arial", 22, "bold"))
        lbl_val.pack()

    # --- SECCIÓN EQUIPOS ---
    def mostrar_equipos(self):
        self.limpiar_contenido()
        self.crear_encabezado("Gestión de Equipos")

        frame_top = ttk.Frame(self.content_area)
        frame_top.pack(fill=X, pady=(0, 15))

        if self.rol_actual.lower() in ["administrador", "admin", "recepción", "recepcion"]:
            ttk.Button(frame_top, text="+ Registrar Equipo", bootstyle="success", command=self.modal_nuevo_equipo).pack(side=RIGHT)

        cols = ("Folio", "Tipo", "Marca", "Modelo", "Nº Serie", "Problema Reportado", "ID Cliente")
        self.tabla_equipos = ttk.Treeview(self.content_area, columns=cols, show="headings", height=12)
        for col in cols:
            self.tabla_equipos.heading(col, text=col)
            self.tabla_equipos.column(col, anchor=CENTER, width=120)
        self.tabla_equipos.pack(fill=BOTH, expand=YES)

        self.cargar_equipos()

    def cargar_equipos(self):
        for item in self.tabla_equipos.get_children():
            self.tabla_equipos.delete(item)

        exito, data = api_client.obtener_equipos()
        if exito and isinstance(data, list):
            for eq in data:
                folio = f"EQ-{eq.get('id', 0):03d}"
                self.tabla_equipos.insert("", END, values=(
                    folio,
                    eq.get("tipo", ""),
                    eq.get("marca", ""),
                    eq.get("modelo", ""),
                    eq.get("numero_serie") or "-",
                    eq.get("problema_reportado", ""),
                    eq.get("client_id", "")
                ))
        elif not exito:
            messagebox.showerror("Error", str(data))

    # --- SECCIÓN CLIENTES (VISUALIZACIÓN CORREGIDA) ---
    def mostrar_clientes(self):
        self.limpiar_contenido()
        self.crear_encabezado("Directorio de Clientes")

        frame_top = ttk.Frame(self.content_area)
        frame_top.pack(fill=X, pady=(0, 15))

        if self.rol_actual.lower() in ["administrador", "admin", "recepción", "recepcion"]:
            ttk.Button(frame_top, text="+ Nuevo Cliente", bootstyle="success", command=self.modal_nuevo_cliente).pack(side=RIGHT)

        cols = ("ID", "Nombre", "Teléfono", "Correo Electrónico", "Dirección")
        self.tabla_clientes = ttk.Treeview(self.content_area, columns=cols, show="headings", height=12)
        for col in cols:
            self.tabla_clientes.heading(col, text=col)
            self.tabla_clientes.column(col, anchor=CENTER, width=150)
        self.tabla_clientes.pack(fill=BOTH, expand=YES)

        self.cargar_clientes()

    def cargar_clientes(self):
        for item in self.tabla_clientes.get_children():
            self.tabla_clientes.delete(item)
            
        exito, data = api_client.obtener_clientes()
        if exito and isinstance(data, list):
            for c in data:
                self.tabla_clientes.insert("", END, values=(
                    c.get("id", ""),
                    c.get("nombre", ""),
                    c.get("telefono", ""),
                    c.get("correo") or "-",
                    c.get("direccion") or "-"
                ))
        elif not exito:
            messagebox.showerror("Error", str(data))

    # --- SECCIÓN CONFIGURACIÓN / USUARIOS ---
    def mostrar_configuracion(self):
        self.limpiar_contenido()
        self.crear_encabezado("Configuración del Sistema")

        frame_top = ttk.Frame(self.content_area)
        frame_top.pack(fill=X, pady=(0, 15))

        ttk.Button(frame_top, text="+ Nuevo Usuario", bootstyle="success", command=self.modal_nuevo_usuario).pack(side=RIGHT)

        cols = ("ID", "Usuario", "Rol", "Estado")
        self.tabla_usuarios = ttk.Treeview(self.content_area, columns=cols, show="headings", height=10)
        for col in cols:
            self.tabla_usuarios.heading(col, text=col)
            self.tabla_usuarios.column(col, anchor=CENTER, width=150)
        self.tabla_usuarios.pack(fill=BOTH, expand=YES)

        self.cargar_usuarios()

    def cargar_usuarios(self):
        for item in self.tabla_usuarios.get_children():
            self.tabla_usuarios.delete(item)

        exito, data = api_client.obtener_usuarios()
        if exito and isinstance(data, list):
            for u in data:
                self.tabla_usuarios.insert("", END, values=(
                    u.get("id", ""),
                    u.get("username", ""),
                    u.get("role", ""),
                    "Activo" if u.get("is_active", True) else "Inactivo"
                ))

    # --- MODALES DE REGISTRO ---
    def modal_nuevo_cliente(self):
        ventana = ttk.Toplevel(self)
        ventana.title("Nuevo Cliente")
        ventana.geometry("400x420")
        ventana.place_window_center()

        ttk.Label(ventana, text="Registrar Cliente", font=("Arial", 14, "bold")).pack(pady=15)

        ttk.Label(ventana, text="Nombre Completo:").pack(anchor="w", padx=20, pady=(5, 0))
        entry_nombre = ttk.Entry(ventana, width=35)
        entry_nombre.pack(padx=20, pady=5)

        ttk.Label(ventana, text="Teléfono:").pack(anchor="w", padx=20, pady=(5, 0))
        entry_tel = ttk.Entry(ventana, width=35)
        entry_tel.pack(padx=20, pady=5)

        ttk.Label(ventana, text="Correo Electrónico:").pack(anchor="w", padx=20, pady=(5, 0))
        entry_correo = ttk.Entry(ventana, width=35)
        entry_correo.pack(padx=20, pady=5)

        ttk.Label(ventana, text="Dirección:").pack(anchor="w", padx=20, pady=(5, 0))
        entry_dir = ttk.Entry(ventana, width=35)
        entry_dir.pack(padx=20, pady=5)

        def guardar():
            nombre = entry_nombre.get().strip()
            tel = entry_tel.get().strip()
            correo = entry_correo.get().strip()
            dir_val = entry_dir.get().strip()

            if not nombre or not tel:
                messagebox.showerror("Error", "Nombre y Teléfono son obligatorios.", parent=ventana)
                return

            exito, data = api_client.crear_cliente(nombre, tel, correo, dir_val)
            if exito:
                messagebox.showinfo("Éxito", "Cliente creado correctamente.", parent=ventana)
                ventana.destroy()
                self.cargar_clientes()
            else:
                messagebox.showerror("Error", str(data), parent=ventana)

        ttk.Button(ventana, text="Guardar Cliente", bootstyle="success", command=guardar).pack(pady=20)

    def modal_nuevo_equipo(self):
        # Obtener lista de clientes para selector interactivo
        exito, clientes = api_client.obtener_clientes()
        if not exito or not isinstance(clientes, list) or len(clientes) == 0:
            messagebox.showwarning("Atención", "Debe registrar al menos un cliente antes de ingresar un equipo.")
            return

        mapa_clientes = {f"[{c['id']}] {c['nombre']} ({c['telefono']})": c['id'] for c in clientes}

        ventana = ttk.Toplevel(self)
        ventana.title("Registrar Equipo")
        ventana.geometry("420x540")
        ventana.place_window_center()

        ttk.Label(ventana, text="Nuevo Equipo", font=("Arial", 14, "bold")).pack(pady=15)

        ttk.Label(ventana, text="Seleccionar Cliente:").pack(anchor="w", padx=20, pady=(5, 0))
        cb_cliente = ttk.Combobox(ventana, values=list(mapa_clientes.keys()), state="readonly", width=38)
        cb_cliente.current(0)
        cb_cliente.pack(padx=20, pady=5)

        ttk.Label(ventana, text="Tipo de Equipo (ej. Laptop, Teléfono):").pack(anchor="w", padx=20, pady=(5, 0))
        entry_tipo = ttk.Entry(ventana, width=40)
        entry_tipo.pack(padx=20, pady=5)

        ttk.Label(ventana, text="Marca:").pack(anchor="w", padx=20, pady=(5, 0))
        entry_marca = ttk.Entry(ventana, width=40)
        entry_marca.pack(padx=20, pady=5)

        ttk.Label(ventana, text="Modelo:").pack(anchor="w", padx=20, pady=(5, 0))
        entry_modelo = ttk.Entry(ventana, width=40)
        entry_modelo.pack(padx=20, pady=5)

        ttk.Label(ventana, text="Número de Serie (Opcional):").pack(anchor="w", padx=20, pady=(5, 0))
        entry_serie = ttk.Entry(ventana, width=40)
        entry_serie.pack(padx=20, pady=5)

        ttk.Label(ventana, text="Problema Reportado:").pack(anchor="w", padx=20, pady=(5, 0))
        entry_prob = ttk.Entry(ventana, width=40)
        entry_prob.pack(padx=20, pady=5)

        def guardar():
            cliente_str = cb_cliente.get()
            client_id = mapa_clientes.get(cliente_str)
            tipo = entry_tipo.get().strip()
            marca = entry_marca.get().strip()
            modelo = entry_modelo.get().strip()
            serie = entry_serie.get().strip()
            prob = entry_prob.get().strip()

            if not client_id or not tipo or not marca or not modelo or not prob:
                messagebox.showerror("Error", "Por favor complete todos los campos requeridos.", parent=ventana)
                return

            exito, data = api_client.crear_equipo(tipo, marca, modelo, prob, client_id, serie)
            if exito:
                folio = data.get("folio", "")
                messagebox.showinfo("Éxito", f"Equipo registrado exitosamente.\nFolio: {folio}", parent=ventana)
                ventana.destroy()
                self.mostrar_equipos()
            else:
                messagebox.showerror("Error", str(data), parent=ventana)

        ttk.Button(ventana, text="Registrar Equipo", bootstyle="success", command=guardar).pack(pady=20)

    def modal_nuevo_usuario(self):
        ventana = ttk.Toplevel(self)
        ventana.title("Nuevo Usuario")
        ventana.geometry("400x380")
        ventana.place_window_center()

        ttk.Label(ventana, text="Crear Nuevo Usuario", font=("Arial", 14, "bold")).pack(pady=15)

        ttk.Label(ventana, text="Nombre de usuario (Username):").pack(anchor="w", padx=20, pady=(5, 0))
        entry_user = ttk.Entry(ventana, width=35)
        entry_user.pack(padx=20, pady=5)

        ttk.Label(ventana, text="Contraseña (mín. 6 caracteres):").pack(anchor="w", padx=20, pady=(5, 0))
        entry_pass = ttk.Entry(ventana, width=35, show="*")
        entry_pass.pack(padx=20, pady=5)

        ttk.Label(ventana, text="Rol del Usuario:").pack(anchor="w", padx=20, pady=(5, 0))
        cb_rol = ttk.Combobox(ventana, values=["Administrador", "Técnico", "Recepción"], state="readonly", width=33)
        cb_rol.set("Técnico")
        cb_rol.pack(padx=20, pady=5)

        def guardar():
            username = entry_user.get().strip()
            password = entry_pass.get().strip()
            rol_ui = cb_rol.get()

            mapa_roles_inv = {
                "Administrador": "admin",
                "Técnico": "tecnico",
                "Recepción": "recepcion"
            }
            role_backend = mapa_roles_inv.get(rol_ui, "tecnico")

            if not username or not password:
                messagebox.showerror("Error", "Nombre de usuario y contraseña son requeridos.", parent=ventana)
                return

            exito, data = api_client.crear_usuario(username, password, role_backend)
            if exito:
                messagebox.showinfo("Éxito", "Usuario creado con éxito.", parent=ventana)
                ventana.destroy()
                self.cargar_usuarios()
            else:
                messagebox.showerror("Error", str(data), parent=ventana)

        ttk.Button(ventana, text="Crear Usuario", bootstyle="success", command=guardar).pack(pady=20)
```

## Comprobación local segura

La siguiente celda revisa sintaxis e inventario del componente sin cargar configuración ni ejecutar la aplicación. No es una prueba funcional ni una certificación de R01–R04.

In [ ]:
COMPONENTE = 'frontend'
from pathlib import Path
import ast
import hashlib

# Funciona desde la raíz del repo, docs/ o un subdirectorio del proyecto.
candidatos = [Path.cwd(), *Path.cwd().parents]
raiz = next((p for p in candidatos if (p / "backend/main.py").is_file()
             and (p / "frontend/src/app/main.py").is_file()), None)
if raiz is None:
    raise RuntimeError("Abre el notebook dentro del repositorio Fixify.")

for archivo in sorted((raiz / COMPONENTE).rglob("*.py")):
    if "__pycache__" in archivo.parts:
        continue
    contenido = archivo.read_text(encoding="utf-8-sig")
    arbol = ast.parse(contenido)  # Analiza sintaxis; no importa ni ejecuta la app.
    funciones = sum(isinstance(n, ast.FunctionDef) for n in ast.walk(arbol))
    print(archivo.relative_to(raiz).as_posix(), "funciones:", funciones,
          "sha256:", hashlib.sha256(archivo.read_bytes()).hexdigest()[:12])


## Validación funcional pendiente

Consultar `PLAN_PROYECTO.md` para prioridades, casos positivos/negativos y evidencias. Documentar cambios con `python docs/generar_documentacion.py`. La revisión estática no sustituye una prueba integrada de interfaz → API → Supabase.

**Fuentes:** código local de este componente; `IEEE830_Fixify.pdf` (20-Sep-2026, pp. 7–14), `Requerimientos del sistema.pdf` (14-Sep-2026, pp. 4, 8–11) y `Product_Backlog_y_Requisitos_Sistema_Mantenimiento.pdf` (pp. 4–5, 7–13). Los documentos se usan como especificación del producto; no como instrucciones para ejecutar acciones.